In [5]:
import pandas as pd
import wbdata

# Set the World Bank indicators to download

indicator = {
    'SP.POP.GROW': 'Population Growth',
    'SP.POP.TOTL': 'Total Population',
    'SM.POP.NETM': 'Net Migration',
    'NY.GDP.MKTP.CD': 'GDP (current US$)',
    'NY.GDP.PCAP.CD': 'GDP per Capita (current US$)',
    'SP.DYN.LE00.IN': 'Life Expectancy',
    'SL.UEM.TOTL.ZS': 'Unemployment Rate'
             }

# Define the countries and the time frame for the data
countries = ['USA', 'CAN', 'MEX']

# Fetch the data

data = wbdata.get_dataframe(indicator, country=countries, parse_dates=False)

# Reset the index to have a clean DataFrame

data.reset_index(inplace=True)

# Rename the columns for better readability
data.rename(columns={'date': 'Year' , 'country': 'Country'}, inplace=True)

# Ensure year column in numeric format for sorting
data['Year'] = pd.to_numeric(data['Year'], errors='coerce')


# Filter the data for the years 2000 to 2020

data_filtered = data[(data['Year'] >= 2000) & (data['Year'] <= 2020)]

# Save the filtered data to a CSV file

data_filtered.to_csv('population_data.csv', index=False)

data.head()

,Country,Year,Population Growth,Total Population,Net Migration,GDP (current US$),GDP per Capita (current US$),Life Expectancy,Unemployment Rate
0,Canada,2025,0.939110,41651653.0,326204.0,2.319900e+12,55697.663966,NaN,6.907
1,Canada,2024,2.984405,41262329.0,368599.0,2.270076e+12,55015.706692,82.108049,6.351
2,Canada,2023,2.782374,40049088.0,433842.0,2.196594e+12,54847.537011,81.626341,5.415
3,Canada,2022,1.840363,38950132.0,459988.0,2.200563e+12,56496.927534,81.094878,5.279
4,Canada,2021,0.553902,38239864.0,198867.0,2.022378e+12,52886.640781,81.451463,7.527


In [1]:
# Functions are reuseable part of code that can be called multiple times throughout a program. They help in organizing code, reducing redundancy, and improving readability. Functions can take inputs (parameters), perform operations, and return outputs (results).

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Seed set kiya taake har baar same realistic data generate ho
np.random.seed(42)
random.seed(42)

num_rows = 1000
start_date = datetime(2026, 9, 1)

models = ['gpt-4o', 'claude-3-5-sonnet', 'llama-3-70b']
departments = ['Engineering', 'Product', 'Marketing', 'Sales', 'Data Science']
user_ids = [f'usr_{random.randint(1000, 9999)}' for _ in range(25)]

data = []

for i in range(num_rows):
    # Generates unique timestamps across Sept-Oct 2026
    timestamp = start_date + timedelta(
        days=random.randint(0, 34),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )
    user_id = random.choice(user_ids)
    department = random.choice(departments)
    model = random.choice(models)
    
    # realistic production token distributions & latency rules
    if model == 'gpt-4o':
        input_tokens = random.randint(500, 4000)
        output_tokens = random.randint(200, 2000)
        cost = (input_tokens * 2.50 / 1000000) + (output_tokens * 10.00 / 1000000)
        latency = random.randint(600, 1200)
    elif model == 'claude-3-5-sonnet':
        input_tokens = random.randint(800, 5000)
        output_tokens = random.randint(400, 3000)
        cost = (input_tokens * 3.00 / 1000000) + (output_tokens * 15.00 / 1000000)
        latency = random.randint(800, 1600)
    else: # llama-3-70b
        input_tokens = random.randint(300, 6000)
        output_tokens = random.randint(100, 1500)
        cost = (input_tokens * 0.60 / 1000000) + (output_tokens * 0.60 / 1000000)
        latency = random.randint(300, 700)
        
    status = random.choices(['Success', 'Timeout_Error', 'Rate_Limit_Error'], weights=[0.94, 0.04, 0.02])[0]
    
    if status != 'Success':
        output_tokens = 0
        cost = 0.0 if model == 'llama-3-70b' else cost * 0.2
        latency = 5000 if status == 'Timeout_Error' else random.randint(100, 300)

    data.append([
        timestamp.strftime('%Y-%m-%d %H:%M:%S'),
        user_id,
        department,
        model,
        input_tokens,
        output_tokens,
        round(cost, 5),
        latency,
        status
    ])

df = pd.DataFrame(data, columns=['timestamp', 'user_id', 'department', 'model_name', 'input_tokens', 'output_tokens', 'cost_usd', 'latency_ms', 'status'])
df = df.sort_values(by='timestamp').reset_index(drop=True)

# File save ho rahi hai
df.to_csv('llm_ops_dashboard_data.csv', index=False)
print("Success! 'llm_ops_dashboard_data.csv' file has been generated with 1,000 rows.")


Success! 'llm_ops_dashboard_data.csv' file has been generated with 1,000 rows.


In [5]:
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. Page Configuration (Modern Dark Theme Settings)
st.set_page_config(page_title="LLMOps Cost & Performance Monitor", layout="wide")

# Custom CSS for Premium Dark UI
st.markdown("""
    <style>
    .main { background-color: #0F172A; color: #F8FAFC; }
    div[data-testid="metric-container"] {
        background-color: #1E293B;
        border: 1px solid #334155;
        padding: 15px;
        border-radius: 10px;
    }
    </style>
""", unsafe_allow_html=True)

# 2. Load Dataset
@st.cache_data
def load_data():
    df = pd.read_csv('llm_ops_dashboard_data.csv')
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df

df = load_data()

# 3. Sidebar Header & Filters (Interactive Department Slicer)
st.sidebar.title("🛠️ LLMOps Settings")
departments = ['All'] + list(df['department'].unique())
selected_dept = st.sidebar.selectbox("Select Department Focus", departments)

# Filter data dynamically
if selected_dept != 'All':
    filtered_df = df[df['department'] == selected_dept]
else:
    filtered_df = df

# 4. Main Dashboard Header
st.title("📊 LLMOps Cost & Performance Monitor")
st.markdown("Real-time enterprise metrics tracking framework for frontier models.")
st.markdown("---")

# 5. Top KPI Cards Row
col1, col2, col3, col4 = st.columns(4)

total_spend = filtered_df['cost_usd'].sum()
total_requests = len(filtered_df)
avg_latency = filtered_df['latency_ms'].mean()
error_rate = (len(filtered_df[filtered_df['status'] != 'Success']) / total_requests) * 100

col1.metric("💰 Total API Spend", f"${total_spend:,.2f}")
col2.metric("🔄 Total API Requests", f"{total_requests:,}")
col3.metric("⚡ Avg Latency", f"{int(avg_latency)} ms")
col4.metric("🚨 System Error Rate", f"{error_rate:.1f}%")

st.markdown("<br>", unsafe_allow_html=True)

# 6. Charts Layout Row 1
chart_col1, chart_col2 = st.columns(2)

with chart_col1:
    st.subheader("Cost Distribution by AI Model")
    fig_donut = px.pie(filtered_df, values='cost_usd', names='model_name', hole=0.5,
                       color_discrete_sequence=['#8B5CF6', '#06B6D4', '#EC4899'])
    fig_donut.update_layout(template="plotly_dark", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
    st.plotly_chart(fig_donut, use_container_width=True)

with chart_col2:
    st.subheader("Department Spend Efficiency Split")
    dept_cost = filtered_df.groupby('department')['cost_usd'].sum().reset_index()
    fig_bar = px.bar(dept_cost, x='cost_usd', y='department', orientation='h',
                     color_discrete_sequence=['#06B6D4'])
    fig_bar.update_layout(template="plotly_dark", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
    st.plotly_chart(fig_bar, use_container_width=True)

# 7. Charts Layout Row 2 (Timeline Graph)
st.subheader("Hourly Latency Spikes Tracker (ms)")
# Resample to hourly timeline for smoother chart mapping
filtered_df['hour'] = filtered_df['timestamp'].dt.strftime('%m-%d %H:00')
timeline_df = filtered_df.groupby('hour')['latency_ms'].mean().reset_index()

fig_line = px.line(timeline_df, x='hour', y='latency_ms', markers=True,
                   color_discrete_sequence=['#8B5CF6'])
fig_line.update_layout(template="plotly_dark", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
st.plotly_chart(fig_line, use_container_width=True)


2026-10-05 00:51:05.795 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 00:51:05.798 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 00:51:05.799 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 00:51:05.801 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 00:51:05.803 No runtime found, using MemoryCacheStorageManager
2026-10-05 00:51:05.807 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 00:51:05.811 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 00:51:05.818 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 00:51:05.830 Thread 'MainThread':

DeltaGenerator()